# Notebook 10 — Convergent and Divergent Evolution

**Module:** 06 — Genetics and Evolution  
**Notebook:** 10 of 12  
**Prerequisites:** NB06 (phylogenetic trees), NB04 (selection), NB03 (mutation rates)  
**Time estimate:** 75 minutes

> dN/dS is the standard metric for detecting selection at the protein-coding level.
> It is used in nearly every comparative genomics paper and appears in Track A
> interview questions: "How would you tell if a gene is under positive selection?"

---
## Step 1 — Motivation

Evolution can drive unrelated species to look identical (convergence: e.g. bat wings
and bird wings) or cause related species to look different (divergence). At the
molecular level, the ratio of non-synonymous to synonymous substitution rates (dN/dS)
reveals whether natural selection has been acting on a gene, and in which direction.
This is one of the most actionable signals in comparative genomics.

---
## Step 3 — Biological Background

### Convergent Evolution: Same Solution, Different Lineages

| Trait | Lineage 1 | Lineage 2 | Molecular basis |
|-------|-----------|-----------|------------------|
| Camera eye | Vertebrates | Cephalopods | Different opsins, inverted retina |
| Echolocation | Bats | Dolphins | Convergent in *Prestin* (DFNB59) |
| C4 photosynthesis | Grasses | Dicots | Same pathway, ~60 independent origins |
| Antifreeze proteins | Arctic fish | Antarctic fish | Non-homologous sequences, same function |

### The dN/dS Ratio (ω)

**dN:** non-synonymous substitution rate (amino acid changes per non-synonymous site)
**dS:** synonymous substitution rate (silent changes per synonymous site)

| ω = dN/dS | Interpretation |
|-----------|----------------|
| ω < 1 | Purifying selection — most amino acid changes are removed |
| ω = 1 | Neutral evolution — amino acid changes fix at same rate as synonymous ones |
| ω > 1 | Positive (diversifying) selection — amino acid changes are favoured |

**Genome-wide patterns:**
- Most genes: ω ≈ 0.1–0.3 (strong purifying selection)
- Rapidly evolving immune genes (MHC): ω ≈ 1.0–5.0 at antigen-binding residues
- Sperm competition genes: among the highest ω in mammalian genomes

### Ancestral Sequence Reconstruction

ML can infer the most likely ancestral amino acid at each node in a tree. Used to:
- Resurrect ancestral proteins for lab experiments (ancestral sequence reconstruction, ASR)
- Identify convergent mutations in separate lineages
- Key paper: Thornton (2001) inferred ancestral steroid receptor specificity

---
## Step 4 — Mathematical Explanation

**Counting synonymous and non-synonymous sites:**

For each codon, count the proportion of changes at each position that would be
synonymous. Summed over all positions of all codons = total synonymous sites S_sites.
Non-synonymous sites N_sites = (codon positions) × L − S_sites.

**Observed changes → corrected rates:**

Count observed synonymous (s_obs) and non-synonymous (n_obs) differences between
two aligned proteins. Apply JC-style correction:

$$d_S = -\frac{3}{4}\ln\left(1 - \frac{4}{3}\cdot\frac{s_{obs}}{S_{sites}}\right)$$
$$d_N = -\frac{3}{4}\ln\left(1 - \frac{4}{3}\cdot\frac{n_{obs}}{N_{sites}}\right)$$
$$\omega = d_N / d_S$$

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product

In [ ]:
# Cell 6.1 — Genetic code and codon table
GENETIC_CODE = {
    'TTT':'F','TTC':'F','TTA':'L','TTG':'L',
    'CTT':'L','CTC':'L','CTA':'L','CTG':'L',
    'ATT':'I','ATC':'I','ATA':'I','ATG':'M',
    'GTT':'V','GTC':'V','GTA':'V','GTG':'V',
    'TCT':'S','TCC':'S','TCA':'S','TCG':'S',
    'CCT':'P','CCC':'P','CCA':'P','CCG':'P',
    'ACT':'T','ACC':'T','ACA':'T','ACG':'T',
    'GCT':'A','GCC':'A','GCA':'A','GCG':'A',
    'TAT':'Y','TAC':'Y','TAA':'*','TAG':'*',
    'CAT':'H','CAC':'H','CAA':'Q','CAG':'Q',
    'AAT':'N','AAC':'N','AAA':'K','AAG':'K',
    'GAT':'D','GAC':'D','GAA':'E','GAG':'E',
    'TGT':'C','TGC':'C','TGA':'*','TGG':'W',
    'CGT':'R','CGC':'R','CGA':'R','CGG':'R',
    'AGT':'S','AGC':'S','AGA':'R','AGG':'R',
    'GGT':'G','GGC':'G','GGA':'G','GGG':'G',
}

def count_synonymous_sites(codon: str) -> float:
    """
    Count the fraction of synonymous sites in a codon.
    For each position, count proportion of single-nucleotide changes that are synonymous.
    """
    bases = 'ACGT'
    ref_aa = GENETIC_CODE.get(codon, '?')
    if ref_aa == '?' or ref_aa == '*':
        return 0.0
    syn_sites = 0.0
    for pos in range(3):
        n_syn = 0
        n_total = 0
        for alt_base in bases:
            if alt_base == codon[pos]:
                continue
            alt_codon = codon[:pos] + alt_base + codon[pos+1:]
            alt_aa = GENETIC_CODE.get(alt_codon, '*')
            if alt_aa != '*':
                n_total += 1
                if alt_aa == ref_aa:
                    n_syn += 1
        if n_total > 0:
            syn_sites += n_syn / n_total
    return syn_sites / 3  # normalise per position

# Test
test_codons = ['CTG', 'ATG', 'TTT', 'GGG', 'TGG']
for codon in test_codons:
    f_syn = count_synonymous_sites(codon)
    print(f"  {codon} ({GENETIC_CODE[codon]}): {f_syn:.3f} synonymous fraction")

In [ ]:
# Cell 6.2 — dN/dS calculation (simplified)
def dnds(seq1: str, seq2: str) -> dict:
    """
    Compute dN/dS between two aligned coding sequences.
    Uses JC-corrected rates. Sequences must be in-frame and same length.

    Returns: dict with dN, dS, omega, and counts.
    """
    assert len(seq1) == len(seq2)
    assert len(seq1) % 3 == 0
    seq1, seq2 = seq1.upper(), seq2.upper()
    n_codons = len(seq1) // 3

    S_sites_total = 0.0
    N_sites_total = 0.0
    n_syn_obs = 0
    n_nonsyn_obs = 0

    for i in range(n_codons):
        c1 = seq1[3*i:3*i+3]
        c2 = seq2[3*i:3*i+3]
        if len(c1) < 3 or len(c2) < 3:
            continue
        if c1 == c2:
            # No difference — still contributes to site counts
            f_syn = count_synonymous_sites(c1)
            S_sites_total += f_syn * 3
            N_sites_total += (1 - f_syn) * 3
            continue

        aa1 = GENETIC_CODE.get(c1, '?')
        aa2 = GENETIC_CODE.get(c2, '?')
        if '?' in (aa1, aa2) or '*' in (aa1, aa2):
            continue

        f_syn = (count_synonymous_sites(c1) + count_synonymous_sites(c2)) / 2
        S_sites_total += f_syn * 3
        N_sites_total += (1 - f_syn) * 3

        if aa1 == aa2:
            n_syn_obs += 1
        else:
            n_nonsyn_obs += 1

    # JC correction
    def jc_correct(obs, sites):
        if sites == 0:
            return 0.0
        p = obs / sites
        if p >= 0.75:
            return np.inf
        return -0.75 * np.log(1 - 4/3 * p)

    dS = jc_correct(n_syn_obs,    S_sites_total)
    dN = jc_correct(n_nonsyn_obs, N_sites_total)
    omega = dN / dS if dS > 0 else np.inf

    return {'dN': dN, 'dS': dS, 'omega': omega,
            'n_syn': n_syn_obs, 'n_nonsyn': n_nonsyn_obs,
            'S_sites': S_sites_total, 'N_sites': N_sites_total}


# Simulate sequences under different selection regimes
rng = np.random.default_rng(42)

def evolve_coding_seq(seq: str, omega_target: float, n_mut: int, rng) -> str:
    """Simulate evolution of a coding sequence with approximate target dN/dS."""
    bases = list('ACGT')
    seq = list(seq)
    n_codons = len(seq) // 3

    for _ in range(n_mut):
        pos = rng.integers(len(seq))
        codon_idx = pos // 3
        codon_pos = pos % 3
        old_codon = ''.join(seq[3*codon_idx:3*codon_idx+3])
        old_aa = GENETIC_CODE.get(old_codon, '?')
        if old_aa in ('?', '*'):
            continue

        new_base = rng.choice([b for b in bases if b != seq[pos]])
        new_codon = old_codon[:codon_pos] + new_base + old_codon[codon_pos+1:]
        new_aa = GENETIC_CODE.get(new_codon, '*')
        if new_aa == '*':
            continue

        is_syn = (new_aa == old_aa)
        # Accept based on omega_target:
        # Synonymous always accepted; non-synonymous accepted with prob min(omega_target,1)
        if is_syn or rng.random() < omega_target:
            seq[pos] = new_base

    return ''.join(seq)

# Start from a random coding sequence
start_seq = ''.join(rng.choice(list(GENETIC_CODE.keys()), 100))
scenarios = [
    ('Purifying (omega=0.1)', 0.1),
    ('Neutral (omega=1.0)', 1.0),
    ('Positive (omega=5.0)', 5.0),
]
print(f"{'Scenario':>30} {'dN':>8} {'dS':>8} {'omega':>8}")
print("-" * 58)
for name, w in scenarios:
    evolved = evolve_coding_seq(start_seq, w, n_mut=50, rng=rng)
    res = dnds(start_seq, evolved)
    print(f"  {name:>28} {res['dN']:>8.4f} {res['dS']:>8.4f} {res['omega']:>8.3f}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — dN/dS distribution + convergent evolution schematic
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Panel 1: dN/dS distribution across gene classes (approximate literature values)
ax = axes[0]
categories = ['Histone H3\n(ultra-conserved)', 'Structural\nproteins', 'Metabolic\nenzymes',
              'Most genes\n(genome avg)', 'Immunity\ngenes', 'Sperm\nproteins',
              'Rapid\nevolution']
omega_vals = [0.002, 0.05, 0.12, 0.22, 0.8, 1.5, 3.0]
colors_dn = ['steelblue' if w < 1 else ('orange' if w < 1.5 else 'tomato') for w in omega_vals]

bars = ax.barh(categories, omega_vals, color=colors_dn, alpha=0.8)
ax.axvline(1.0, color='black', ls='--', lw=1.5, label='ω=1 (neutral)')
ax.set_xlabel('dN/dS (ω)')
ax.set_title('dN/dS across gene classes:\nmost genes under purifying selection')
ax.legend()

for bar, val in zip(bars, omega_vals):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
            f'{val}', va='center', fontsize=8)

# Panel 2: Convergent mutations on phylogeny (schematic)
ax = axes[1]
ax.set_xlim(-0.1, 1.1); ax.set_ylim(-0.1, 1.1)
ax.axis('off')
ax.set_title('Convergent evolution — echolocation gene (Prestin):\nbats and dolphins share same amino acid changes', fontsize=10)

# Draw a simple 4-tip tree
tree_coords = {
    'Bat1': (0.9, 0.85), 'Bat2': (0.9, 0.65),
    'Dolphin': (0.9, 0.35), 'Cow': (0.9, 0.15),
    'NodeBats': (0.7, 0.75), 'NodeEcholocators': None,
    'NodeCetaceans': (0.7, 0.25),
    'Root': (0.2, 0.5)
}
# Draw edges
edges = [
    ('Root', (0.2, 0.5), (0.4, 0.75)),
    ('Root', (0.2, 0.5), (0.4, 0.25)),
    ('Left', (0.4, 0.75), (0.7, 0.75)),
    ('Right', (0.4, 0.25), (0.7, 0.25)),
    ('Bat1', (0.7, 0.75), (0.9, 0.85)),
    ('Bat2', (0.7, 0.75), (0.9, 0.65)),
    ('Dolphin', (0.7, 0.25), (0.9, 0.35)),
    ('Cow', (0.7, 0.25), (0.9, 0.15)),
]
for _, p1, p2 in edges:
    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], 'k-', lw=2)

# Add labels
for name, xy in [('Bat 1', (0.9, 0.85)), ('Bat 2', (0.9, 0.65)),
                  ('Dolphin', (0.9, 0.35)), ('Cow', (0.9, 0.15))]:
    ax.text(xy[0]+0.01, xy[1], name, va='center', fontsize=9)

# Mark convergent mutations on both echolocating lineages
for pos, label in [((0.55, 0.75), '★ Prestin N7T'), ((0.55, 0.26), '★ Prestin N7T')]:
    ax.text(pos[0], pos[1], label, fontsize=8, color='tomato',
            ha='center', style='italic')

ax.text(0.5, 0.55, 'Same amino acid change\nin unrelated lineages\n→ Convergent evolution',
        ha='center', va='center', fontsize=8, color='gray',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7))

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `dnds(seq1, seq2)` from scratch for two 12-codon sequences. Verify
   that a sequence identical to itself gives omega = undefined (0/0) and two
   sequences differing only in synonymous changes give omega ≈ 0.
2. The gene FOXP2 has a very low dN/dS ≈ 0.06 across mammals, but shows evidence
   of positive selection specifically in the human lineage. How can a gene show
   both purifying selection overall and positive selection in one branch? Name the
   PAML model that tests for this.
3. Echolocation evolved independently in bats and dolphins. Studies found convergent
   substitutions in the Prestin gene. What statistical test would you use to determine
   if these convergent mutations are more frequent than expected by chance?
4. Ancestral sequence reconstruction: given a 4-species alignment and a phylogeny,
   describe the steps to infer the ancestral amino acid at a specific position at
   the root. What information is required beyond the alignment?

---
## Quiz — Active Recall

1. What does dN/dS > 1 tell you about a gene?
2. A gene has dN/dS = 0.03. What kind of selection is acting? Why might this be?
3. Define convergent evolution at the molecular level.
4. Name two biological examples of convergent evolution.
5. What is ancestral sequence reconstruction used for?

---
## Reflection

**Date completed:** ____________________

> *[You find that a gene you are studying has dN/dS = 0.95 across mammals but 2.8
> specifically in the primate lineage. What does this suggest biologically, and
> what software would you use to test it formally?]*

---
*Next: `11_mini_project_wf_vs_hardy_weinberg.ipynb`*